# 18xB — Outcome-blind trading signal release

This stage applies the development-frozen strategies to the locked May holdout and June probability release. It reads only outcome-free model probabilities and outcome-free raw market prices. Each row records either a frozen one-share YES signal or no trade. Realised HKO outcomes are unavailable to this stage.

**Revision v2.** The unused drawdown helper is corrected for consistency by including the initial portfolio value of zero. Blind signals are unchanged.

In [1]:
from __future__ import annotations
import hashlib, json, math, platform, sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/'.git').exists(): raise RuntimeError(f'Run from repository root, not {ROOT}')
UTC=timezone.utc

def sha(path:Path)->str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()

def parse_bool(s:pd.Series,name:str)->pd.Series:
    if pd.api.types.is_bool_dtype(s): return s.astype(bool)
    out=s.astype(str).str.strip().str.lower().map({'true':True,'false':False,'1':True,'0':False,'yes':True,'no':False})
    if out.isna().any(): raise ValueError(f'Cannot parse Boolean {name}: {s[out.isna()].drop_duplicates().tolist()}')
    return out.astype(bool)

def verify_manifest(path:Path):
    m=pd.read_csv(path); failures=[]
    for r in m.itertuples(index=False):
        p=ROOT/r.path
        if not p.is_file(): failures.append(f'MISSING {r.path}'); continue
        if sha(p)!=r.sha256: failures.append(f'HASH {r.path}')
        if p.stat().st_size!=int(r.size_bytes): failures.append(f'SIZE {r.path}')
    if failures: raise AssertionError(f'Manifest failed {path}:\\n'+'\\n'.join(failures))

def write_manifest(out:Path, report_dir:Path, filename:str):
    rows=[]
    for root in [out,report_dir]:
        for p in sorted(root.rglob('*')):
            if p.is_file() and p.name!=filename:
                rows.append({'path':str(p.relative_to(ROOT)),'size_bytes':p.stat().st_size,'sha256':sha(p)})
    pd.DataFrame(rows).to_csv(out/filename,index=False)

def save_frame(frame:pd.DataFrame,path:Path):
    x=frame.copy()
    for c in x.columns:
        if pd.api.types.is_datetime64_any_dtype(x[c]):
            if getattr(x[c].dt,'tz',None) is not None: x[c]=x[c].astype('string')
            else: x[c]=x[c].dt.strftime('%Y-%m-%d')
    x.to_csv(path,index=False)

def max_drawdown(daily:pd.Series)->float:
    if daily.empty: return float('nan')
    cumulative=daily.sort_index().cumsum().to_numpy(dtype=float)
    running_peak=np.maximum.accumulate(np.concatenate(([0.0],cumulative)))[1:]
    return float(np.min(cumulative-running_peak))

STEP='18xB'; MAN=ROOT/'data/manual/18x_outcome_free_market_prices'; XA=ROOT/'data/processed/18xA_development_trading_selection'; WC=ROOT/'data/processed/18wC_blind_calibrated_probability_release'
MARKET=MAN/'18x_outcome_free_market_price_panel.csv'; META=MAN/'18x_outcome_free_market_price_metadata.json'; REG=XA/'18xA_selected_strategy_registry.csv'; XA_SUM=XA/'18xA_summary.json'; XA_MAN=XA/'18xA_sha256_manifest.csv'; RELEASE=WC/'18wC_blind_probability_release.csv'; WC_SUM=WC/'18wC_summary.json'; WC_MAN=WC/'18wC_sha256_manifest.csv'
OUT=ROOT/'data/processed/18xB_blind_trading_signal_release'; REPORT=ROOT/'reports/18xB_blind_trading_signal_release'; OUT.mkdir(parents=True,exist_ok=True); REPORT.mkdir(parents=True,exist_ok=True)
for p in [MARKET,META,REG,XA_SUM,XA_MAN,RELEASE,WC_SUM,WC_MAN]:
    if not p.is_file(): raise FileNotFoundError(p)
for p in [XA_MAN,WC_MAN]: verify_manifest(p)
if json.loads(XA_SUM.read_text()).get('verdict')!='PASS' or json.loads(WC_SUM.read_text()).get('verdict')!='PASS': raise AssertionError('Upstream release not PASS')
market=pd.read_csv(MARKET,dtype={'market_id':str},low_memory=False); registry=pd.read_csv(REG); release=pd.read_csv(RELEASE,dtype={'market_id':str},low_memory=False)
for d in [market,release]: d['event_date']=pd.to_datetime(d.event_date,errors='raise')
forbidden={'hko_daily_max_c','Y_event_int','Y_no_int','residual_c','forecast_error_c','market_binary_brier','market_binary_log_score'}
if forbidden.intersection(market.columns) or forbidden.intersection(release.columns): raise AssertionError('Blind signal input contains outcomes')
parts=[]
for r in registry.itertuples(index=False):
    q=release.loc[release.candidate_id.eq(r.candidate_id)&release.probability_variant.eq(r.probability_variant)&release.decision_rule.eq(r.decision_rule)].copy(); q['strategy_role']=r.strategy_role; q['selected_threshold']=r.threshold; q['primary_cost_per_share']=r.primary_cost_per_share; q['market_staleness_cap_hours']=r.market_staleness_cap_hours; parts.append(q)
prob=pd.concat(parts,ignore_index=True)
mcols=['event_date','market_id','decision_rule','p_market','decision_cutoff_utc','selected_price_timestamp_utc','price_staleness_hours']
key=['strategy_role','candidate_id','probability_variant','event_date','decision_rule']
opp=prob.merge(market[mcols],on=['event_date','market_id','decision_rule'],how='inner',validate='many_to_one')
counts=opp.groupby(key).size(); complete=counts[counts.eq(11)].reset_index()[key]; opp=opp.merge(complete.assign(complete_market_book=True),on=key,how='inner')
stale=opp.groupby(key).price_staleness_hours.max().rename('book_max_staleness_hours').reset_index(); opp=opp.merge(stale,on=key,validate='many_to_one'); opp=opp.loc[opp.book_max_staleness_hours.le(opp.market_staleness_cap_hours)].copy(); opp['edge']=opp.p_model-opp.p_market
signals=opp.sort_values(key+['edge','p_model','p_market','market_id'],ascending=[True,True,True,True,True,False,False,True,True],kind='mergesort').groupby(key,as_index=False).head(1).reset_index(drop=True); signals['trade_flag']=signals.edge.ge(signals.selected_threshold); signals['position_size_shares']=np.where(signals.trade_flag,1.0,0.0); signals['assumed_entry_price']=signals.p_market; signals['assumed_primary_cost']=np.where(signals.trade_flag,signals.primary_cost_per_share,0.0); signals['outcome_blind_signal']=True
if len(opp)!=1265 or len(signals)!=115 or int(signals.trade_flag.sum())!=97: raise AssertionError(f'Unexpected blind counts: opp={len(opp)} signals={len(signals)} trades={signals.trade_flag.sum()}')
expected={('PRIMARY_OVERALL','INTERNAL_HOLDOUT'):(10,7),('PRIMARY_OVERALL','EXTERNAL_TEST'):(30,26),('GAUSSIAN_PROCESS_FAMILY','INTERNAL_HOLDOUT'):(10,5),('GAUSSIAN_PROCESS_FAMILY','EXTERNAL_TEST'):(27,22),('TREE_FAMILY','INTERNAL_HOLDOUT'):(10,9),('TREE_FAMILY','EXTERNAL_TEST'):(28,28)}
for k,(books,trades) in expected.items():
    g=signals.loc[signals.strategy_role.eq(k[0])&signals.evaluation_block.eq(k[1])]
    if len(g)!=books or int(g.trade_flag.sum())!=trades: raise AssertionError(f'Coverage differs {k}')
if forbidden.intersection(signals.columns): raise AssertionError('Blind signals contain forbidden fields')
checks=pd.DataFrame([{'check':'blind_inputs_outcome_free','passed':not bool(forbidden.intersection(signals.columns)),'detail':'no realised outcome field','blocking':True},{'check':'blind_contract_opportunity_rows_1265','passed':len(opp)==1265,'detail':str(len(opp)),'blocking':True},{'check':'blind_signal_books_115','passed':len(signals)==115,'detail':str(len(signals)),'blocking':True},{'check':'blind_trade_signals_97','passed':int(signals.trade_flag.sum())==97,'detail':str(int(signals.trade_flag.sum())),'blocking':True},{'check':'raw_prices_used','passed':np.allclose(signals.assumed_entry_price,signals.p_market),'detail':'not normalised','blocking':True},{'check':'outcome_blind_signal_flag','passed':signals.outcome_blind_signal.all(),'detail':'all rows','blocking':True}])
if not checks.passed.all(): raise AssertionError(checks.loc[~checks.passed].to_string(index=False))
issues=pd.DataFrame(columns=['issue_level','issue_code','strategy_role','event_date','decision_rule','detail','blocking'])
for name,frame in {'blind_trade_opportunity_panel':opp,'blind_trading_signal_panel':signals,'blind_signal_coverage_summary':signals.groupby(['strategy_role','candidate_id','probability_variant','decision_rule','evaluation_block'],as_index=False).agg(opportunity_books=('event_date','size'),trade_signals=('trade_flag','sum'),mean_edge=('edge','mean'),max_staleness_hours=('book_max_staleness_hours','max')),'integrity_checks':checks,'issues':issues}.items(): save_frame(frame,OUT/f'18xB_{name}.csv')
protocol={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','source_strategy_registry_hash':sha(REG),'signal_rule':'selected family candidate, probability variant, decision rule and threshold frozen in 18xA; choose highest raw edge in each complete book','outcomes_loaded':False,'market_price_source':'outcome-free raw selected YES price panel','market_prices_normalised':False,'signal_rows':115,'trade_signals':97}
(OUT/'18xB_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
summary={'step':STEP,'generated_at_utc':datetime.now(UTC).isoformat(),'verdict':'PASS','blind_contract_opportunity_rows':1265,'blind_signal_books':115,'blind_trade_signals':97,'holdout_signal_books':30,'holdout_trade_signals':21,'external_signal_books':85,'external_trade_signals':76,'outcomes_loaded':False,'issue_rows':0,'integrity_checks_passed':int(checks.passed.sum()),'integrity_checks_total':len(checks)}
(OUT/'18xB_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
pd.DataFrame([{'input_role':'18xA_selected_strategy_registry','path':str(REG.relative_to(ROOT)),'rows':len(registry),'sha256':sha(REG)},{'input_role':'18wC_blind_probability_release','path':str(RELEASE.relative_to(ROOT)),'rows':len(release),'sha256':sha(RELEASE)},{'input_role':'outcome_free_market_price_panel','path':str(MARKET.relative_to(ROOT)),'rows':len(market),'sha256':sha(MARKET)}]).to_csv(OUT/'18xB_source_inventory.csv',index=False)
(OUT/'18xB_environment.json').write_text(json.dumps({'generated_at_utc':datetime.now(UTC).isoformat(),'python':sys.version,'platform':platform.platform(),'pandas':pd.__version__,'numpy':np.__version__,'revision':'v2'},indent=2),encoding='utf-8')
(REPORT/'18xB_blind_trading_signal_release_report.md').write_text('# 18xB blind trading signal release\n\n**PASS**\n\nThe 115 frozen opportunity rows and 97 trade signals contain no realised HKO outcome or market score. Entry prices are raw observed pre-cutoff YES-price fill proxies.\n',encoding='utf-8')
write_manifest(OUT,REPORT,'18xB_sha256_manifest.csv'); print(json.dumps(summary,indent=2)); print('18xB PASS')

{
  "step": "18xB",
  "generated_at_utc": "2026-07-22T15:21:46.630410+00:00",
  "verdict": "PASS",
  "blind_contract_opportunity_rows": 1265,
  "blind_signal_books": 115,
  "blind_trade_signals": 97,
  "holdout_signal_books": 30,
  "holdout_trade_signals": 21,
  "external_signal_books": 85,
  "external_trade_signals": 76,
  "outcomes_loaded": false,
  "issue_rows": 0,
  "integrity_checks_passed": 6,
  "integrity_checks_total": 6
}
18xB PASS
